# Toyset device probe

Answers one question: on this worker, is the accelerator faster than the
worker's own CPU for the toyset's actual work?

It was measured on an Apple M2 that the accelerator lost to its own CPU by
11.3x per phase-2 step, because a step is 90 points through a dim-8 MLP and
the dispatch costs more than the arithmetic. This notebook asks the same
question of whatever card the worker was given, and reports both halves so
the comparison is this worker's own and not a cross-machine ratio.

In [1]:
#!/usr/bin/env python3
"""The first code cell of every notebook a job may run — copied byte for byte.

One question, answered once: WHERE IS THE REPOSITORY THIS NOTEBOOK RUNS
AGAINST. Every later cell reads `ROOT` and none of them asks again.

Opened by a person on their own machine, this answers it exactly the way
these notebooks always have. A notebook lives at `<repo>/<Name>/Notebooks/`,
so the repository is two directories above the working directory. Nothing was
handed over, nothing is checked, and the behaviour is unchanged.

Started by a runner on a remote worker, it does not answer it by looking
around. The runner exports the directory it cloned the pinned commit into and
the commit it pinned; this cell reads both, and PROVES the checkout is at that
commit before returning it.

Guessing is the whole reason this cell exists. Under the runner the kernel's
working directory is the runner's own and the clone sits one level inside it,
so two directories up is two levels ABOVE the working directory — a directory
that EXISTS on any worker. The path resolves, the insert succeeds, and the run
dies later with a missing module naming a package, never with the wrong root.
The one fact worth having is the one the failure never mentions.

Three refusals, and each one is a refusal rather than a fallback because this
is the path that spends metered quota:

- **Half a handoff** — one variable present and the other missing. Something
  built this environment and got it half right; that is precisely the state a
  fallback cannot tell apart from a laptop, and the fallback resolves a
  directory that exists everywhere.
- **A root that is not a checkout** — the declared directory is missing, or
  holds no readable `HEAD`. There is nothing to compare against, and an
  unproven root is what this cell was written to stop being acceptable.
- **A checkout at a different commit** — the declared commit and the one on
  disk disagree. A job that runs the wrong commit RUNS: it returns numbers
  shaped exactly like the right ones, and nothing downstream can tell.

Absent means LOCAL. It never means "work it out".

Importable and independently testable, the way the runner's own two cells
are: every function is pure and takes its environment and its working
directory as arguments, and only the last line — the binding a notebook cell
exists to perform — reads the real ones.
"""
import os
from pathlib import Path

#: The directory a runner cloned the pinned commit into. Forge-owned and
#: deliberately generic: this is the contract between a runner and the
#: notebook it starts, not a name borrowed from any one repository.
CLONE_ROOT_ENV = "FORGE_CLONE_ROOT"

#: The commit that clone was pinned to, exported beside it so this cell can
#: check rather than trust. A root on its own would only move the guess one
#: step: a directory handed over is still a directory nobody proved.
CLONE_COMMIT_ENV = "FORGE_CLONE_COMMIT"

#: How far above a notebook's own directory the repository sits when nobody
#: hands anything over: `<repo>/<Name>/Notebooks` -> `<repo>`.
LOCAL_ROOT_DEPTH = 1


def head_commit(root):
    """The commit `root`'s `HEAD` names, read straight out of `.git`.

    No subprocess, deliberately. A notebook cell that shells out needs a git
    binary on a worker's PATH that nothing here declared, and every checker
    that reads these notebooks then has to decide whether running the cell is
    safe — a cell that binds the repository is the last one that may be
    skipped for that reason.

    A pinned checkout is detached: the runner fetches a commit and checks out
    what it fetched, never a branch, so `HEAD` holds the raw commit and this
    is a one-line read. A symbolic `HEAD` is resolved anyway — through the
    loose ref and then `packed-refs` — so pointing these variables at an
    ordinary checkout gets an answer instead of a refusal that would be about
    the file format rather than about the commit.

    Returns `None` when there is nothing to read. The caller turns that into
    a refusal; this function never decides.
    """
    git = Path(root) / ".git"
    if git.is_file():
        pointer = git.read_text(encoding="utf-8").strip()
        if not pointer.startswith("gitdir:"):
            return None
        git = Path(root) / pointer[len("gitdir:"):].strip()
    if not git.is_dir():
        return None
    head = git / "HEAD"
    if not head.is_file():
        return None
    text = head.read_text(encoding="utf-8").strip()
    if not text.startswith("ref:"):
        return text or None
    ref = text[len("ref:"):].strip()
    loose = git / ref
    if loose.is_file():
        return loose.read_text(encoding="utf-8").strip() or None
    packed = git / "packed-refs"
    if packed.is_file():
        for line in packed.read_text(encoding="utf-8").splitlines():
            if line.startswith("#") or line.startswith("^"):
                continue
            fields = line.split()
            if len(fields) == 2 and fields[1] == ref:
                return fields[0]
    return None


def resolve_repository_root(environ=None, cwd=None, head_reader=head_commit):
    """The repository this notebook runs against, or a refusal saying why.

    `environ` and `cwd` are arguments so the whole decision can be driven
    without a kernel, a clone or a worker; the binding at the bottom of this
    cell passes the real ones.
    """
    environ = os.environ if environ is None else environ
    here = Path.cwd() if cwd is None else Path(cwd)
    declared_root = environ.get(CLONE_ROOT_ENV)
    declared_commit = environ.get(CLONE_COMMIT_ENV)

    if not declared_root and not declared_commit:
        return here.parents[LOCAL_ROOT_DEPTH]

    if not declared_root or not declared_commit:
        raise RuntimeError(
            "half a handoff: {0}={1!r} and {2}={3!r}. Both name the pinned "
            "checkout this notebook must run against, and one without the "
            "other is an environment somebody built and got half right. "
            "Falling back to the local layout here would resolve a directory "
            "that exists on any machine and is not this repository.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))

    root = Path(declared_root)
    found = head_reader(root) if root.is_dir() else None
    if found is None:
        raise RuntimeError(
            "{0}={1!r} is not a readable checkout: no commit could be read "
            "from its HEAD. {2} declares {3!r}, and there is nothing here to "
            "check it against.".format(
                CLONE_ROOT_ENV, declared_root,
                CLONE_COMMIT_ENV, declared_commit))
    if found.lower() != declared_commit.lower():
        raise RuntimeError(
            "the checkout at {0}={1!r} is at commit {2}, and {3} declares "
            "{4}. A job that runs a commit nobody asked for RUNS, and the "
            "numbers it returns look exactly like the right ones.".format(
                CLONE_ROOT_ENV, declared_root, found,
                CLONE_COMMIT_ENV, declared_commit))
    return root


ROOT = resolve_repository_root()


In [2]:
import os, sys, time, json, platform
import torch

sys.path.insert(0, str(ROOT / 'src'))
print('ROOT         ', ROOT)
print('commit       ', os.environ.get('FORGE_CLONE_COMMIT', '(local)'))
print('modo         ', os.environ.get('FORGE_RUN_MODE', '(local)'))

HW = {
    'platform': platform.platform(),
    'cpuCount': os.cpu_count(),
    'torch': torch.__version__,
    'cudaAvailable': torch.cuda.is_available(),
    'cudaName': (torch.cuda.get_device_name(0)
                 if torch.cuda.is_available() else None),
    'cudaCapability': (list(torch.cuda.get_device_capability(0))
                       if torch.cuda.is_available() else None),
    'mpsAvailable': torch.backends.mps.is_available(),
    'commit': os.environ.get('FORGE_CLONE_COMMIT'),
    'runMode': os.environ.get('FORGE_RUN_MODE'),
}
for k, v in HW.items():
    print(f'{k:16s} {v}')

ACCEL = ('cuda' if torch.cuda.is_available()
         else ('mps' if torch.backends.mps.is_available() else None))
print(f'\nACCEL            {ACCEL}')
if ACCEL is None:
    print('SIN ACELERADOR: la comparacion tiene un solo lado y el job lo dice')

ROOT          /Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation
commit        (local)
modo          (local)
platform         macOS-26.5.2-arm64-arm-64bit
cpuCount         8
torch            2.13.0
cudaAvailable    False
cudaName         None
cudaCapability   None
mpsAvailable     True
commit           None
runMode          None

ACCEL            mps


## 1. Where the crossover is on this worker

A square matmul at six sizes. The size where the accelerator stops losing is
the only number that says whether this workload belongs on it: every matrix
the toyset forms is smaller than 100.

In [3]:
def rate(dev, n, reps=200):
    d = torch.device(dev)
    a = torch.randn(n, n, device=d); b = torch.randn(n, n, device=d)
    for _ in range(10): c = a @ b
    if dev == 'cuda': torch.cuda.synchronize()
    elif dev == 'mps': torch.mps.synchronize()
    t0 = time.perf_counter()
    for _ in range(reps): c = a @ b
    if dev == 'cuda': torch.cuda.synchronize()
    elif dev == 'mps': torch.mps.synchronize()
    return (time.perf_counter() - t0) / reps

CROSSOVER = []
print(f"{'n':>6} {'flops':>14} {'cpu':>11} {'accel':>11}   winner")
print('-' * 64)
for n in (8, 32, 128, 512, 2048, 4096):
    c = rate('cpu', n)
    a = rate(ACCEL, n) if ACCEL else float('nan')
    if ACCEL:
        who = f'CPU   {a/c:5.1f}x' if c < a else f'ACCEL {c/a:5.1f}x'
    else:
        who = 'cpu only'
    CROSSOVER.append({'n': n, 'flops': 2*n**3, 'cpuSeconds': c,
                      'accelSeconds': a})
    print(f'{n:>6} {2*n**3:>14,} {c*1e6:>9.1f}us {a*1e6:>9.1f}us   {who}')

     n          flops         cpu       accel   winner
----------------------------------------------------------------


     8          1,024       1.0us      18.7us   CPU    19.1x
    32         65,536       1.4us      18.5us   CPU    13.0x
   128      4,194,304       6.0us      38.9us   CPU     6.4x
   512    268,435,456     200.1us     153.3us   ACCEL   1.3x


  2048 17,179,869,184   14744.0us    6726.5us   ACCEL   2.2x


  4096 137,438,953,472  123963.8us   54945.8us   ACCEL   2.3x


## 2. One real cell, at full scale, on each device

The same cell both devices ran on the M2: one arm (`TGL`), lunas, batchnorm,
seed 0, with the hyperparameters the full-scale artifact chose. The work is
identical on both -- phase 2 has no stopping rule, so it runs its 100 epochs
whole -- which is what makes the two times comparable.

In [4]:
from MIL_CREDA_Benchmark import toy_run as T

E = T.FULL
ART = ROOT / 'MIL-CREDA' / 'Results' / 'Toyset' / 'full' / 'barrido.json'
hp = json.load(open(ART))
hyper = dict(hp['hp1']['lunas']); hyper.update(hp['hp2']['lunas'])
print('escala', E.name, '| epochs_1', E.epochs_1, '| epochs_2', E.epochs_2,
      '| puntos/dominio', E.points_per_domain, '| dim', hyper['dim'])

familia = T.build_family('lunas', 0, E)
CELL = {}
for name in (['cpu'] + ([ACCEL] if ACCEL else [])):
    t0 = time.perf_counter()
    fila = T.run_cell(familia, 'batchnorm', 0, E, hyper,
                      torch.device(name), arms=('TGL',))
    wall = time.perf_counter() - t0
    r = fila['rows']['TGL']
    CELL[name] = {
        'wallSeconds': wall,
        'secondsPerPhase1Epoch': r['secondsPerPhase1Epoch'],
        'secondsPerPhase2Step': r['secondsPerPhase2Step'],
        'phase2Epochs': len(r['phase2EpochPass']),
        'targetAccuracy': r['targetAccuracy'],
    }
    print(f"{name:5s} {wall:7.1f}s  s/epoca_f1={r['secondsPerPhase1Epoch']:.5f}  "
          f"s/paso_f2={r['secondsPerPhase2Step']:.5f}  "
          f"epocas_f2={len(r['phase2EpochPass'])}  "
          f"acc={r['targetAccuracy']:.4f}", flush=True)

/Users/diego/Proyectos/papersmith-ai/implementations/Domain_Adaptation/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


escala full | epochs_1 100 | epochs_2 100 | puntos/dominio 4050 | dim 8


cpu      16.7s  s/epoca_f1=0.02524  s/paso_f2=0.00967  epocas_f2=100  acc=0.5556


mps     185.3s  s/epoca_f1=0.13967  s/paso_f2=0.11182  epocas_f2=100  acc=0.3333


## 3. What the whole flow would cost here

`work_counts`' own ceilings at full scale, multiplied by the rates THIS worker
measured. An upper bound, for the reason `cost_report` already states: the
early stop of phase 1 ends it before its declared length.

In [5]:
FORECAST = {}
for name, m in CELL.items():
    t1 = E.epochs_1 * m['secondsPerPhase1Epoch']
    t2 = E.epochs_2 * T.STEPS_PER_EPOCH_DECLARED * m['secondsPerPhase2Step']
    s1 = 27 * (25*t1 + t1 + 25*t2) + 90 * (t1 + t2)
    s25 = 30*(t1+2*t2) + 270*(t1+t2) + 30*(t1+2*t2) + 30*(t1+3*t2)
    FORECAST[name] = {'section1Hours': s1/3600, 'sections2to5Hours': s25/3600,
                      'wholeFlowHours': (s1+s25)/3600}
    print(f'{name:5s}  seccion 1 {s1/3600:6.2f} h   secciones 2-5 {s25/3600:6.2f} h'
          f'   FLUJO {(s1+s25)/3600:6.2f} h')

print()
print('=== VEREDICTO ===')
if ACCEL and ACCEL in CELL:
    r = CELL['cpu']['secondsPerPhase2Step'] / CELL[ACCEL]['secondsPerPhase2Step']
    print(f'por paso de fase 2 el acelerador es {r:.2f}x la CPU de este worker')
    print('el acelerador GANA' if r > 1 else 'el acelerador PIERDE contra su propia CPU')
else:
    print('sin acelerador en este worker')

cpu    seccion 1   3.84 h   secciones 2-5   2.32 h   FLUJO   6.16 h
mps    seccion 1  41.09 h   secciones 2-5  25.25 h   FLUJO  66.34 h

=== VEREDICTO ===
por paso de fase 2 el acelerador es 0.09x la CPU de este worker
el acelerador PIERDE contra su propia CPU


In [6]:
import pathlib
shard = {'probe': 'toyset-device', 'hardware': HW,
         'crossover': CROSSOVER, 'cell': CELL, 'forecast': FORECAST}
out = pathlib.Path('.') / 'shard.json'   # el directorio de trabajo del runner
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(json.dumps(shard, indent=2, default=str))
print('escrito', out, out.stat().st_size, 'bytes')

escrito shard.json 1955 bytes
